# ETL 04b - Gold: agregados TLC 2026

Objetivo: leer la capa **silver** (viajes limpios, fila por fila) y construir tablas **gold**: datos ya agregados y listos para consumir. La app `05_plot_tlc.py` las lee directamente en lugar de recorrer decenas de millones de filas cada vez que se mueve un filtro.

```text
bronze/  →  silver/  →  gold/
crudo       limpio      agregado para un uso concreto
```

## Dependencias

```bash
conda install pandas pyarrow
```

In [1]:
from pathlib import Path
import urllib.request

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds

PROJECT_DIR = Path.cwd()
BASE_DIR = PROJECT_DIR if PROJECT_DIR.name == "etl-04" else PROJECT_DIR / "etl-04"
BRONZE_DIR = BASE_DIR / "data" / "bronze" / "tlc"
SILVER_DIR = BASE_DIR / "data" / "silver" / "tlc"
GOLD_DIR = BASE_DIR / "data" / "gold" / "tlc"
CLEAN_DIR = SILVER_DIR / "tlc_2026_clean"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

## 1. Leer silver

Leemos solo las columnas que necesitamos (formato columnar: el resto ni se toca). La partición `record_type`/`year_month` viene de los nombres de carpeta.

In [3]:
partition_schema = pa.schema(
    [
        ("record_type", pa.string()),
        ("year_month", pa.string()),
    ]
)
try:
    silver = ds.dataset(
        CLEAN_DIR,
        format="parquet",
        partitioning=ds.partitioning(partition_schema, flavor="hive"),
    )
except Exception as e:
    print(f"Error al leer el dataset silver: {e}")
    raise

columns = ["record_type", "year_month", "pickup_location_id", "dropoff_location_id", "duration_min"]
trips = silver.to_table(columns=columns)
print(f"Filas en silver: {trips.num_rows:,}")

Error al leer el dataset silver: Unable to merge: Field record_type has incompatible types: string vs large_string


ArrowTypeError: Unable to merge: Field record_type has incompatible types: string vs large_string

## 2. Dimensión de zonas

Los viajes solo traen el número de zona (`location_id`). Para que gold sea útil sin más joins, le agregamos barrio (`borough`) y nombre de zona con la tabla oficial de TLC. Es un dato de referencia, así que lo guardamos en **bronze** si todavía no está.

In [ ]:
LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
LOOKUP_PATH = BRONZE_DIR / "taxi_zone_lookup.csv"

if not LOOKUP_PATH.exists():
    urllib.request.urlretrieve(LOOKUP_URL, LOOKUP_PATH)

zones = pd.read_csv(LOOKUP_PATH).rename(
    columns={"LocationID": "location_id", "Borough": "borough", "Zone": "zone"}
)
zones["zone_label"] = zones["borough"] + " / " + zones["zone"]
zones = zones[["location_id", "borough", "zone", "service_zone", "zone_label"]]
zones.head()

## 3. Tabla `trips_by_zone`

Una fila por **tipo de viaje × mes × pickup/dropoff × zona**, con la cantidad de viajes y la duración media. Los viajes sin zona (muchos `fhv` no la informan) no se pueden ubicar y se excluyen.

In [ ]:
def count_by_zone(table: pa.Table, location_column: str, location_kind: str) -> pd.DataFrame:
    with_zone = table.filter(pc.is_valid(table[location_column]))
    grouped = with_zone.group_by(["record_type", "year_month", location_column]).aggregate(
        [([], "count_all"), ("duration_min", "mean")]
    )
    out = grouped.to_pandas().rename(
        columns={
            location_column: "location_id",
            "count_all": "trips",
            "duration_min_mean": "avg_duration_min",
        }
    )
    out.insert(2, "location_kind", location_kind)
    return out


zone_trips = pd.concat(
    [
        count_by_zone(trips, "pickup_location_id", "pickup"),
        count_by_zone(trips, "dropoff_location_id", "dropoff"),
    ],
    ignore_index=True,
)

trips_by_zone = zone_trips.merge(zones, on="location_id", how="left")
for col in ["borough", "zone", "service_zone"]:
    trips_by_zone[col] = trips_by_zone[col].fillna("Sin lookup")
trips_by_zone["zone_label"] = trips_by_zone["zone_label"].fillna(
    "LocationID " + trips_by_zone["location_id"].astype(str)
)

trips_by_zone = (
    trips_by_zone[
        [
            "record_type",
            "year_month",
            "location_kind",
            "location_id",
            "borough",
            "zone",
            "service_zone",
            "zone_label",
            "trips",
            "avg_duration_min",
        ]
    ]
    .astype({"location_id": "int64", "trips": "int64"})
    .sort_values(["record_type", "year_month", "location_kind", "trips"], ascending=[True, True, True, False])
    .reset_index(drop=True)
)
trips_by_zone["avg_duration_min"] = trips_by_zone["avg_duration_min"].round(2)

print(f"trips_by_zone: {len(trips_by_zone):,} filas")
trips_by_zone.head()

## 4. Tabla `trips_by_borough`

Se deriva de la anterior sumando por barrio. La duración media se pondera por cantidad de viajes (el promedio de promedios no sería correcto).

In [ ]:
weighted = trips_by_zone.assign(total_duration_min=trips_by_zone["avg_duration_min"] * trips_by_zone["trips"])

trips_by_borough = (
    weighted.groupby(["record_type", "year_month", "location_kind", "borough"], as_index=False)
    .agg(trips=("trips", "sum"), total_duration_min=("total_duration_min", "sum"))
)
trips_by_borough["avg_duration_min"] = (
    trips_by_borough["total_duration_min"] / trips_by_borough["trips"]
).round(2)
trips_by_borough = trips_by_borough.drop(columns="total_duration_min").sort_values(
    ["record_type", "year_month", "location_kind", "trips"], ascending=[True, True, True, False]
).reset_index(drop=True)

print(f"trips_by_borough: {len(trips_by_borough):,} filas")
trips_by_borough.head()

## 5. Escribir gold

Son tablas chicas (miles de filas, no millones), así que alcanza con un Parquet por tabla, sin particionar.

In [ ]:
outputs = {
    "trips_by_zone": trips_by_zone,
    "trips_by_borough": trips_by_borough,
}

for name, table in outputs.items():
    path = GOLD_DIR / f"{name}.parquet"
    table.to_parquet(path, index=False)
    print(f"{path.relative_to(BASE_DIR)}: {len(table):,} filas, {path.stat().st_size / 1024:.1f} KB")

## 6. Verificación

Gold tiene que cuadrar con silver: por cada tipo de viaje, mes y pickup/dropoff, la suma de `trips` debe ser igual a la cantidad de filas de silver que tienen zona.

In [ ]:
silver_counts = []
for kind, column in [("pickup", "pickup_location_id"), ("dropoff", "dropoff_location_id")]:
    counts = (
        trips.filter(pc.is_valid(trips[column]))
        .group_by(["record_type", "year_month"])
        .aggregate([([], "count_all")])
        .to_pandas()
        .rename(columns={"count_all": "silver_rows"})
    )
    counts["location_kind"] = kind
    silver_counts.append(counts)

check = pd.concat(silver_counts).merge(
    trips_by_zone.groupby(["record_type", "year_month", "location_kind"], as_index=False)["trips"]
    .sum()
    .rename(columns={"trips": "gold_trips"}),
    on=["record_type", "year_month", "location_kind"],
    how="outer",
)
check["coincide"] = check["silver_rows"] == check["gold_trips"]
display(check.sort_values(["record_type", "year_month", "location_kind"]).reset_index(drop=True))
assert check["coincide"].all(), "Gold no cuadra con silver"
print("OK: gold cuadra con silver")

In [ ]:
# Las 10 zonas con más pickups de yellow en todos los meses
(
    trips_by_zone.query("record_type == 'yellow' and location_kind == 'pickup'")
    .groupby("zone_label", as_index=False)["trips"]
    .sum()
    .sort_values("trips", ascending=False)
    .head(10)
)

Resultado esperado:

- `etl-04/data/gold/tlc/trips_by_zone.parquet`: viajes por tipo, mes, pickup/dropoff y zona.
- `etl-04/data/gold/tlc/trips_by_borough.parquet`: lo mismo agregado por barrio.

Siguiente paso: `streamlit run etl-04/05_plot_tlc.py` lee estas tablas.